In [ ]:
import time
import pandas as pd
from sklearn.metrics import r2_score
from nba_api.stats.endpoints import leaguegamelog

Fetching 2021-22...
Fetching 2022-23...
Fetching 2023-24...
Fetching 2024-25...
Total games: 4915
      game_id   season       date home_team away_team  home_points  \
0  0022100002  2021-22 2021-10-19       LAL       GSW          114   
1  0022100001  2021-22 2021-10-19       MIL       BKN          127   
2  0022100009  2021-22 2021-10-20       NOP       PHI           97   
3  0022100012  2021-22 2021-10-20       PHX       DEN           98   
4  0022100013  2021-22 2021-10-20       POR       SAC          121   
5  0022100006  2021-22 2021-10-20       TOR       WAS           83   
6  0022100011  2021-22 2021-10-20       UTA       OKC          107   
7  0022100004  2021-22 2021-10-20       DET       CHI           88   
8  0022100008  2021-22 2021-10-20       MIN       HOU          124   
9  0022100010  2021-22 2021-10-20       SAS       ORL          123   

   away_points  home_won  away_won  
0          121       0.0       1.0  
1          104       1.0       0.0  
2          117      

In [ ]:
seasons = ['2021-22', '2022-23', '2023-24', '2024-25']
all_games = []

for season in seasons:
    print(f"Fetching {season}...")
    log = leaguegamelog.LeagueGameLog(season=season, season_type_all_star='Regular Season')
    df = log.get_data_frames()[0]
    df['season'] = season
    all_games.append(df)
    time.sleep(1)

full_df = pd.concat(all_games, ignore_index=True)

home = full_df[full_df['MATCHUP'].str.contains(r'vs\.')].copy()
away = full_df[full_df['MATCHUP'].str.contains('@')].copy()

scores = home.merge(away, on='GAME_ID', suffixes=('_home', '_away'))

scores = pd.DataFrame({
    'game_id':     scores['GAME_ID'],
    'season':      scores['season_home'],
    'date':        pd.to_datetime(scores['GAME_DATE_home']),
    'home_team':   scores['TEAM_ABBREVIATION_home'],
    'away_team':   scores['TEAM_ABBREVIATION_away'],
    'home_points': scores['PTS_home'],
    'away_points': scores['PTS_away'],
    'home_won':    (scores['WL_home'] == 'W').astype(float),
    'away_won':    (scores['WL_away'] == 'W').astype(float),
})

print(f"Total games: {len(scores)}")
print(scores.head(10))

In [23]:
def zmr2(ytrue, ypred):
    return 1 - ((ytrue - ypred)**2).sum() / (ytrue ** 2).sum()

In [24]:
zmr2(scores["home_won"] - 0.5, [0.05] * len(scores))

0.011200406917598826

In [20]:
scores["home_won"]

0       0.0
1       1.0
2       0.0
3       0.0
4       0.0
       ... 
4910    1.0
4911    0.0
4912    1.0
4913    1.0
4914    1.0
Name: home_won, Length: 4915, dtype: float64

In [14]:
scores.describe()

,date,home_points,away_points,home_won,away_won
count,4915,4915.000000,4915.000000,4915.000000,4915.000000
mean,2023-07-16 18:09:18.128178944,114.343845,112.328179,0.553001,0.446999
min,2021-10-19 00:00:00,73.000000,67.000000,0.000000,0.000000
25%,2022-04-10 00:00:00,106.000000,104.000000,0.000000,0.000000
50%,2023-04-09 00:00:00,114.000000,112.000000,1.000000,0.000000
75%,2024-04-14 00:00:00,123.000000,121.000000,1.000000,1.000000
max,2025-04-13 00:00:00,175.000000,176.000000,1.000000,1.000000
std,NaN,12.579425,12.617891,0.497234,0.497234


In [ ]:

# Fetch player-level game logs for the same seasons
all_player_games = []

for season in seasons:
    print(f"Fetching players {season}...")
    log = leaguegamelog.LeagueGameLog(
        season=season,
        season_type_all_star='Regular Season',
        player_or_team_abbreviation='P'
    )
    df = log.get_data_frames()[0]
    df['season'] = season
    all_player_games.append(df)
    time.sleep(1)

player_df = pd.concat(all_player_games, ignore_index=True)
player_df = player_df.rename(columns={
    'GAME_ID': 'game_id',
    'PLAYER_ID': 'player_id',
    'PLAYER_NAME': 'player_name',
    'TEAM_ABBREVIATION': 'team',
    'PTS': 'points',
    'REB': 'rebounds',
    'AST': 'assists',
    'MIN': 'minutes',
    'WL': 'wl',
})

print(f"Total player-game rows: {len(player_df)}")
print(player_df[['game_id', 'player_name', 'team', 'points', 'rebounds', 'assists', 'wl', 'season']].head(10))


In [ ]:

# Merge player data with game scores
# Each player row gets home/away context from scores

player_games = player_df[['game_id', 'player_id', 'player_name', 'team', 'points', 'rebounds', 'assists', 'minutes', 'wl', 'season']].copy()

player_games = player_games.merge(
    scores[['game_id', 'date', 'home_team', 'away_team', 'home_points', 'away_points', 'home_won', 'away_won']],
    on='game_id',
    how='left'
)

player_games['is_home'] = (player_games['team'] == player_games['home_team']).astype(int)
player_games['won'] = (player_games['wl'] == 'W').astype(float)

print(f"Player-game rows with game context: {len(player_games)}")
print(player_games[['date', 'player_name', 'team', 'is_home', 'points', 'won']].head(10))
